In [7]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

In [8]:
df = pd.read_csv("merged_trends_influenza_wide.csv")
X_vif = df[["grip", "virus gripa", "simptomi gripa"]].dropna()
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])
]
print(vif_data)

          Feature       VIF
0            grip  2.037420
1     virus gripa  3.681459
2  simptomi gripa  3.342342


In [9]:
df = pd.read_csv("merged_trends_influenza_wide.csv")
df["week_sin"] = np.sin(2 * np.pi * df["ISO_WEEK"] / 52)  # Sezonska komponenta
X = df[["grip", "virus gripa", "simptomi gripa", "week_sin"]].shift(1)
y = df["INF_ALL"]
mask = ~X.isna().any(axis=1) & ~y.isna()
X, y = X[mask], y[mask]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = LinearRegression()
model.fit(X_train, y_train)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
print(f"Train R^2: {model.score(X_train, y_train):.3f}")
print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.3f}")
print(f"Test R^2: {model.score(X_test, y_test):.3f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.3f}")
print("\nKoeficijenti modela:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"{feature}: {coef:.3f}")
print(f"Intercept: {model.intercept_:.3f}")

Train R^2: 0.556
Train RMSE: 28.725
Test R^2: 0.711
Test RMSE: 20.200

Koeficijenti modela:
grip: 0.556
virus gripa: 1.021
simptomi gripa: -0.059
week_sin: 8.591
Intercept: -1.180


In [10]:
df = pd.read_csv("merged_trends_influenza_wide.csv")
df["week_sin"] = np.sin(2 * np.pi * df["ISO_WEEK"] / 52)
X = df[["grip", "virus gripa", "simptomi gripa", "week_sin"]].shift(1)
y = df["INF_ALL"]
mask = ~X.isna().any(axis=1) & ~y.isna()
X, y = X[mask], y[mask]
model = LinearRegression()
scores = cross_val_score(model, X, y, cv=5, scoring="r2")
print(f"Cross-validation R^2: {scores.mean():.3f} ± {scores.std():.3f}")

Cross-validation R^2: -0.029 ± 0.736


In [5]:
print(f"Test set size: {len(test_df)}")
print(
    f"Test set INF_ALL mean: {test_df['y'].mean():.3f}, std: {test_df['y'].std():.3f}"
)

Test set size: 97
Test set INF_ALL mean: 11.247, std: 26.260
